# Random Number Generator on a Local Simulator

&copy; 2026 by [Damir Cavar](http://damir.cavar.me/)

This is an example of a random number generator using two quantum bits and a Bell circuit, updated for Qiskit 2.x.

Prerequisites can be installed using the following code:

In [ ]:
!pip install -U "qiskit>=2.0"
!pip install -U "qiskit-ibm-runtime>=0.30"
!pip install -U "qiskit-aer>=0.15"

# Local Simulated Random Generator

In Qiskit 2.x we run circuits on a backend through the V2 `Sampler` primitive; `backend.run()` is no longer the supported path for real-device execution. The same code works against `FakeManilaV2` (a noise-model-backed fake backend) by passing the fake backend to `SamplerV2` via `mode=`.

We need the following imports:

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler
from qiskit_aer import AerSimulator

We define a 2-qubit Bell circuit with measurements into a classical register:

In [2]:
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

Use the `AerSimulator` backend and produce an ISA circuit with the preset pass manager:

In [3]:
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(circuit)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

We can print and visualize the circuit:

In [4]:
print(isa_qc)

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 


In [8]:
print(result[0].data) # .c.get_counts())

DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=2>))


Print the result. In Qiskit 2.x the V2 `Sampler` returns per-PUB results and the measurement bitstring counts live under the classical register's name (here, `c`):

In [10]:
pub_result = result[0]
counts = pub_result.data.meas.get_counts()
print('Result:', counts)

Result: {'00': 526, '11': 498}


(C) 2026 by [Damir Cavar](http://damir.cavar.me/)